In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 8
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.381265994028912



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 1 =========================================
15.96544365474766

Trial 2 =========================================
14.634438850702471

Trial 3 =========================================
15.379316147905444

Trial 4 =========================================
15.398815881610467

Trial 5 =========================================
14.270643754922629

Trial 6 =========================================
13.832889903456428

Trial 7 =========================================
14.464753463577027

Trial 8 =========================================
14.432558563423642

Trial 9 =========================================
14.442547649453441

Trial 10 =========================================
13.915268984929293

Trial 11 =========================================
18.799214207430072

Trial 12 =========================================
17.512807681599746

Trial 13 =========================================
13.601388824062472

Trial 14 =========================================
14.691713861718899

Trial 15 =======

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 26 =========================================
14.96900614512621

Trial 27 =========================================
16.103926213230565

Trial 28 =========================================
14.366364699276854

Trial 29 =========================================
15.39361614083206

Trial 30 =========================================
13.432834255061401

Trial 31 =========================================
17.253164699859457

Trial 32 =========================================
14.612702692828226

Trial 33 =========================================
15.378563465345811

Trial 34 =========================================
15.37141975495517

Trial 35 =========================================
14.96455926430048

Trial 36 =========================================
16.240230266058088



...

Trial 72 =========================================
16.178140343034027



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 73 =========================================
16.066850288704593

Trial 74 =========================================
15.377081818274183

Trial 75 =========================================
13.151255825844341

Trial 76 =========================================
13.867567549735334

Trial 77 =========================================
16.131011308441305

Trial 78 =========================================
13.39112493452334

Trial 79 =========================================
16.92115207890516

Trial 80 =========================================
14.945226906822615

Trial 81 =========================================
16.20094442821251

Trial 82 =========================================
15.388527431791395

Trial 83 =========================================
14.63528791004143



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 84 =========================================
15.917187571098594

Trial 85 =========================================
17.483045404147873

Trial 86 =========================================
14.828904303693538

Trial 87 =========================================
17.487975242243902

Trial 88 =========================================
15.336090581407714

Trial 89 =========================================
15.899338820049651

Trial 90 =========================================
13.454386544601794

Trial 91 =========================================
16.19418857408698

Trial 92 =========================================
13.737616757904778

Trial 93 =========================================
18.581652672863797

Trial 94 =========================================
14.128778894075767

Trial 95 =========================================
13.891782961905458

Trial 96 =========================================
14.936823290317157

Trial 97 =========================================
14.50942266453333

Trial 98

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.821548381976754
Avg = 15.239826522087363
Std = 1.2681235296832765


In [6]:
print(y_max_arr.tolist())

[15.381265994028912, 15.96544365474766, 14.634438850702471, 15.379316147905444, 15.398815881610467, 14.270643754922629, 13.832889903456428, 14.464753463577027, 14.432558563423642, 14.442547649453441, 13.915268984929293, 18.799214207430072, 17.512807681599746, 13.601388824062472, 14.691713861718899, 14.4643996788939, 13.690564556602707, 18.768214266007394, 14.141028773483217, 15.973956350206723, 14.38705594972279, 16.07344665360494, 16.05216002229705, 18.821548381976754, 13.897843477329548, 15.301850070666845, 14.96900614512621, 16.103926213230565, 14.366364699276854, 15.39361614083206, 13.432834255061401, 17.253164699859457, 14.612702692828226, 15.378563465345811, 15.37141975495517, 14.96455926430048, 16.240230266058088, 14.206114556110109, 14.46341856342101, 16.13318420868108, 13.34011486518669, 14.285664123154515, 13.964373043723397, 15.391169162622244, 14.270548320075054, 18.792960936482324, 14.235337385933077, 14.444514542081135, 15.378656821836625, 16.205628368936786, 14.384785603

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-8.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)